# L8 demo: the split that inflates a score, then versioning

We reuse the NASA C-MAPSS turbofan data from L7 and predict remaining useful life
(RUL). The point of this notebook is not the model. It is that the same model and the
same features can report two very different scores depending only on how the rows are
split, and that only one of those scores is the one the model will earn on a new engine.

Then we make the honest result reproducible: content-hash the data the way DVC does,
and log the runs to MLflow tagged with that hash.

> Data: [C-MAPSS FD001](https://www.nasa.gov/intelligent-systems-division/discovery-and-systems-health/pcoe/pcoe-data-set-repository/),
> 100 run-to-failure engines, carried over from L7.

## Run this first on Colab

Colab starts from its own preinstalled environment rather than this course's `uv`
environment, so run the cell below before anything else. It installs what this
notebook needs and Colab does not already have. Outside Colab it does nothing, so
you can run it or skip it.


In [ ]:
# Run this first on Colab. Anywhere else this cell does nothing.
#
# Only genuinely missing packages are installed, so Colab's own versions of
# everything it already ships are left alone.
#
# Generated by tools/colab_setup.py from this notebook's imports. Edit that.
import importlib.util
import subprocess
import sys

REQUIREMENTS = {
    "mlflow": "mlflow",
    "numpy": "numpy",
    "pandas": "pandas",
    "sklearn": "scikit-learn",
}


def _missing(module):
    try:
        return importlib.util.find_spec(module) is None
    except ModuleNotFoundError:  # the parent package is absent
        return True


if "google.colab" in sys.modules:
    need = sorted({pip for mod, pip in REQUIREMENTS.items() if _missing(mod)})
    if need:
        print("installing:", " ".join(need))
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *need], check=True)
    print("Colab setup done." if need else "Colab: nothing to install.")


## 1. Load C-MAPSS FD001

One row per engine per cycle: 3 operational settings and 21 sensor channels, with the
failure cycle known in the training data. Fetched once and cached under `data/`.

In [ ]:
import io
import urllib.request
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

CACHE = Path('data/CMAPSS')
URL = ('https://phm-datasets.s3.amazonaws.com/NASA/'
       '6.+Turbofan+Engine+Degradation+Simulation+Data+Set.zip')
NEEDED = ['train_FD001.txt', 'test_FD001.txt', 'RUL_FD001.txt', 'readme.txt']

N_SETTINGS, N_SENSORS = 3, 21
COLUMNS = (['unit', 'cycle']
           + [f'setting{i + 1}' for i in range(N_SETTINGS)]
           + [f'sensor{i + 1}' for i in range(N_SENSORS)])


def fetch():
    CACHE.mkdir(parents=True, exist_ok=True)
    if all((CACHE / f).exists() for f in NEEDED):
        return
    print('downloading', URL)
    with urllib.request.urlopen(URL) as r:
        payload = r.read()
    outer = zipfile.ZipFile(io.BytesIO(payload))
    inner_name = next(n for n in outer.namelist() if n.lower().endswith('.zip'))
    inner = zipfile.ZipFile(io.BytesIO(outer.read(inner_name)))
    for name in NEEDED:
        (CACHE / name).write_bytes(inner.read(name))


fetch()
train = pd.read_csv(CACHE / 'train_FD001.txt', sep=r'\s+', header=None, names=COLUMNS)
train[['unit', 'cycle']] = train[['unit', 'cycle']].astype(int)
print(f"{len(train):,} rows, {train['unit'].nunique()} engines")
train.head(3)

## 2. Per-engine features and a clipped RUL target

Every rolling feature is computed **within an engine** (`groupby('unit')`), never across
the boundary between one engine and the next, exactly as L7 insisted. Six of the 21
sensors are constant and drop out. The target is remaining cycles, clipped at 125, which
is a modeling choice this dataset conventionally makes.

In [ ]:
WINDOW, RUL_CAP, SEED = 5, 125, 0


def build_features(df, rolling=True):
    df = df.sort_values(['unit', 'cycle']).reset_index(drop=True)
    sensors = [c for c in df.columns if c.startswith('sensor')]
    live = [c for c in sensors if df[c].nunique() > 1]
    g = df.groupby('unit', group_keys=False)
    feats = {'cycle': df['cycle']}
    for c in live:
        feats[c] = df[c]
        if rolling:
            feats[f'{c}_rmean'] = g[c].transform(
                lambda s: s.rolling(WINDOW, min_periods=1).mean())
            feats[f'{c}_rstd'] = g[c].transform(
                lambda s: s.rolling(WINDOW, min_periods=1).std()).fillna(0.0)
    X = pd.DataFrame(feats)
    life = g['cycle'].transform('max')
    y = (life - df['cycle']).clip(upper=RUL_CAP).to_numpy()
    groups = df['unit'].to_numpy()
    return X, y, groups, live


X, y, groups, live = build_features(train)
print(f'{X.shape[1]} features from {len(live)} live sensors; target clipped at {RUL_CAP}')

## 3. The same model, two splits

We score one RandomForest with 5-fold cross-validation two ways. A **random** split
shuffles the rows. A **grouped** split keeps each engine wholly in train or in test. The
only difference between the two calls is the splitter.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, GroupKFold, cross_val_score

model = RandomForestRegressor(n_estimators=100, n_jobs=-1, random_state=SEED)
rmse = 'neg_root_mean_squared_error'

random_rmse = -cross_val_score(
    model, X, y, cv=KFold(5, shuffle=True, random_state=SEED), scoring=rmse).mean()
group_rmse = -cross_val_score(
    model, X, y, cv=GroupKFold(5), groups=groups, scoring=rmse).mean()

print(f'random row split (leaks) : {random_rmse:5.2f} cycles RMSE')
print(f'per-unit GroupKFold      : {group_rmse:5.2f} cycles RMSE')
print(f'the leak makes the model look {group_rmse / random_rmse:.2f}x better than it is')

The random split reports the lower error, so it is the one a careless review would
ship. The grouped split reports what the model will actually earn on an engine it has
never seen. Same rows, same model; only the split differs.

## 4. Why the random split leaks

Consecutive cycles of one engine are near-duplicates, so a shuffled split puts almost
every engine on both sides of the line. We can count it directly for one split.

In [ ]:
from sklearn.model_selection import train_test_split

idx = np.arange(len(X))
tr, te = train_test_split(idx, test_size=0.2, random_state=SEED)
both = set(groups[tr]) & set(groups[te])
print(f'{len(both)} of {train["unit"].nunique()} engines appear in BOTH train and test')
print('so the model is graded on cycles almost identical to ones it trained on')

The honest splitters keep an entity whole. `GroupKFold` and `GroupShuffleSplit` split
by engine; `TimeSeriesSplit` trains on past cycles and tests on later ones. All three
take the same one-line shape as the calls above.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit, TimeSeriesSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
tr_g, te_g = next(gss.split(X, y, groups))
print(f'GroupShuffleSplit holds out {len(set(groups[te_g]))} whole engines, '
      f'{len(set(groups[tr_g]) & set(groups[te_g]))} shared')
print(f'TimeSeriesSplit gives {TimeSeriesSplit(n_splits=5).get_n_splits()} past/future folds')

## 5. Version the data by content

Reproducing a result means pinning the data as precisely as the code. DVC does this by
**content hashing**: it hashes the file's bytes and tracks that hash in git. We can
compute the same hash ourselves to see what it stores.

In [ ]:
import hashlib

Path('artifacts').mkdir(exist_ok=True)
feature_path = Path('artifacts/features.parquet')
X.assign(rul=y, unit=groups).to_parquet(feature_path)

digest = hashlib.md5(feature_path.read_bytes()).hexdigest()
print(f'{feature_path}  ->  md5 {digest}')
print('this hash is exactly what a .dvc metafile records for the file')

### Narrated DVC workflow (commands shown, not run here)

DVC is a command-line tool. In the assignment you run these; the point to see now is that
the `.dvc` metafile git tracks is tiny, and the data itself goes to a cache and a
**local remote that is just a directory**, so no cloud account is needed.

```bash
dvc init
dvc remote add -d local ../dvc-store    # a plain directory
dvc add artifacts/features.parquet      # hashes + caches the file
git add artifacts/features.parquet.dvc .gitignore
dvc push                                # copy bytes to the remote
```

The `features.parquet.dvc` file that git then tracks looks like this, and its `md5`
matches the digest printed above:

```yaml
outs:
  - md5: <the digest above>
    path: features.parquet
```

A `dvc.yaml` records the pipeline that produced the file, so `dvc repro` rebuilds it from
raw data and reruns only the stages whose inputs changed:

```yaml
stages:
  featurize:
    cmd: python featurize.py
    deps: [data/CMAPSS/train_FD001.txt, featurize.py]
    outs: [artifacts/features.parquet]
```

## 6. Track the runs, tagged with the data version

Now the honest and leaky scores become reproducible facts. Each MLflow run records the
split, the score, and the **data hash**, so a run can be tied back to the exact bytes it
used. MLflow stores this in a local SQLite file, with no server to start.

In [ ]:
import mlflow

mlflow.set_tracking_uri('sqlite:///mlflow.db')
mlflow.set_experiment('l08-rul-splits')

for split_name, score in [('random_row', random_rmse), ('per_unit_group', group_rmse)]:
    with mlflow.start_run(run_name=split_name):
        mlflow.log_param('split', split_name)
        mlflow.log_param('data_md5', digest)
        mlflow.log_param('model', 'RandomForest(100)')
        mlflow.log_metric('rmse_cycles', score)

runs = mlflow.search_runs(experiment_names=['l08-rul-splits'])
print(runs[['params.split', 'metrics.rmse_cycles', 'params.data_md5']].to_string(index=False))

## 7. Data-centric iteration

Improve the model by improving the data, with the model and the honest split held fixed,
so any change in the score is attributable to the data. The change here is a **label**
decision. C-MAPSS RUL is conventionally clipped at 125 cycles, which encodes that an
engine's health is roughly flat until late in life. We train on the raw remaining-cycle
count and on the clipped version, score both on the same clipped target (the quantity we
actually care about) under the same `GroupKFold`, and log each as a run.

In [ ]:
from sklearn.model_selection import cross_val_predict

def rmse(a, b):
    return float(np.sqrt(((a - b) ** 2).mean()))

d = train.sort_values(['unit', 'cycle']).reset_index(drop=True)
rul_raw = (d.groupby('unit')['cycle'].transform('max') - d['cycle']).to_numpy()
y_true = np.clip(rul_raw, 0, RUL_CAP)          # scored on this, both ways
gkf = GroupKFold(5)

scores = {}
for name, y_train in [('unclipped_rul', rul_raw), ('clipped_rul_125', y_true)]:
    path = Path(f'artifacts/features_{name}.parquet')
    X.assign(rul=y_train, unit=groups).to_parquet(path)
    d_md5 = hashlib.md5(path.read_bytes()).hexdigest()
    pred = np.clip(cross_val_predict(model, X, y_train, cv=gkf, groups=groups), 0, RUL_CAP)
    scores[name] = rmse(y_true, pred)
    with mlflow.start_run(run_name=name):
        mlflow.log_param('label_scheme', name)
        mlflow.log_param('data_md5', d_md5)
        mlflow.log_metric('rmse_cycles', scores[name])

gain = scores['unclipped_rul'] - scores['clipped_rul_125']
print(f"train on unclipped RUL : {scores['unclipped_rul']:5.2f} cycles (honest split, scored on clipped)")
print(f"train on clipped RUL   : {scores['clipped_rul_125']:5.2f} cycles")
print(f'clipping the label improved the honest RMSE by {gain:.2f} cycles, same model and split')

---

## Takeaway

The split decides whether your score means anything. A random split of grouped, ordered
data leaks, and it flatters the model by about 37% here; a per-unit split reports what the
model earns on a new engine. Version the data by content so a run pins its exact inputs,
log the data hash beside the score, and then improve the data against a fixed model so the
gain is attributable and reproducible. Assignment **A4** has you put the C-MAPSS features
under DVC, implement a correct split, and quantify the leak, so its second half starts here.